In [31]:
!pip install bs4 yfinance python-dotenv python-dateutil --upgrade certifi openpyxl xlrd==2.0.1 --upgrade


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
import certifi
import requests
SESSION = requests.Session()
SESSION.verify = certifi.where()

In [33]:
from dotenv import load_dotenv
import os

load_dotenv("API.env")  # busca automáticamente el archivo .env

FRED_API_KEY = os.getenv("FRED_API_KEY")

if not FRED_API_KEY:
    raise ValueError("FRED_API_KEY no encontrada. Crear archivo .env con la clave.")

print("FRED API cargada correctamente.")

FRED API cargada correctamente.


In [34]:
import os
import time
import warnings
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse
import re

# ---------- OPTIONAL: load API.env / .env ----------
try:
    from dotenv import load_dotenv
    if Path("API.env").exists():
        load_dotenv("API.env")
    elif Path(".env").exists():
        load_dotenv(".env")
except Exception:
    # OK si no está instalado python-dotenv, siempre que la key esté en env vars
    pass

# ---------- CONFIG ----------
RUN_DATE = datetime.now().strftime("%Y-%m-%d")
START_DATE = "2015-01-01"
END_DATE = datetime.now().strftime("%Y-%m-%d")

BASE_ORIG = Path("Bases originales")
RAW_DIR = Path("data/raw")  # opcional (si querés guardar dumps / logs)
BASE_ORIG.mkdir(exist_ok=True, parents=True)
RAW_DIR.mkdir(exist_ok=True, parents=True)

# Si querés panel diario homogéneo + forward fill (series mensuales/semanales):
MAKE_DAILY_PANEL = True  # <- ponelo True si querés daily + ffill

# ---------- HTTP SESSION ----------
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; DataPipeline/1.0)"
})

# Silenciar warnings de SSL cuando usemos verify=False para BCU
try:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
except Exception:
    pass


import requests
import urllib3

def download_file(url, out_path, *, timeout=120, verify=True):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if verify is False:
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    r = SESSION.get(url, stream=True, timeout=timeout, verify=verify)
    r.raise_for_status()

    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

# =========================
# HELPERS
# =========================
def should_disable_ssl_verify(url: str) -> bool:
    """
    Para destrabar el error de certificados (SSLCertVerificationError) en algunos entornos Windows/corporativos.
    Solo se desactiva para BCU.
    """
    host = urlparse(url).netloc.lower()
    return "bcu.gub.uy" in host


def download_file(url: str, out_path: Path, timeout: int = 120, max_tries: int = 3) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    verify_ssl = not should_disable_ssl_verify(url)
    last_err = None

    for k in range(max_tries):
        try:
            with SESSION.get(
                url,
                stream=True,
                timeout=timeout,
                allow_redirects=True,
                verify=verify_ssl
            ) as r:
                r.raise_for_status()
                with open(out_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            return out_path
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (2 ** k))

    raise RuntimeError(f"Falló descarga: {url} -> {out_path}. Error: {last_err}")


def pick_first_xlsx_link(page_url: str) -> str:
    """
    Busca el primer link .xlsx en una página. Devuelve URL absoluta.
    """
    html = SESSION.get(page_url, timeout=60).text
    soup = BeautifulSoup(html, "html.parser")
    links = [a.get("href") for a in soup.select("a[href]") if a.get("href")]
    xlsx = [u for u in links if ".xlsx" in u.lower()]
    if not xlsx:
        raise RuntimeError(f"No encontré .xlsx en {page_url}")
    return urljoin(page_url, xlsx[0])


def pick_ascoma_latest_download(page_url: str) -> str:
    """
    ASCOMA suele usar links /download/.../ que llevan al archivo o a una página con el link final.
    Devuelve URL absoluta al primer /download/ (en la práctica suele ser el más reciente).
    """
    html = SESSION.get(page_url, timeout=60).text
    soup = BeautifulSoup(html, "html.parser")
    links = [a.get("href") for a in soup.select("a[href]") if a.get("href")]
    dl = [u for u in links if "/download/" in u]
    if not dl:
        raise RuntimeError("No encontré links /download/ en ASCOMA")
    return urljoin(page_url, dl[0])


def to_daily_ffill(df: pd.DataFrame, date_col: str, start=START_DATE, end=END_DATE) -> pd.DataFrame:
    """
    Reindex a calendario diario completo [start,end] y forward-fill para todas las columnas (excepto fecha).
    Útil si querés un panel diario homogéneo (como describiste).
    """
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce").dt.normalize()
    out = out.dropna(subset=[date_col]).sort_values(date_col)

    idx = pd.date_range(pd.to_datetime(start), pd.to_datetime(end), freq="D")
    out = out.set_index(date_col).reindex(idx).ffill().reset_index().rename(columns={"index": date_col})
    return out

In [35]:
# =========================
# 1) FRED (requiere API key)
# =========================
FRED_API_KEY = os.getenv("FRED_API_KEY", "").strip()
if not FRED_API_KEY:
    raise RuntimeError(
        "Falta FRED_API_KEY. Ponela en API.env (recomendado) o .env o como variable de entorno.\n"
        "Ejemplo API.env:\nFRED_API_KEY=tu_clave"
    )

FRED_SERIES = {
    "EFFR": "FedFunds",
    "UNRATE": "UnemploymentRate",
    "CPIAUCSL": "CPI",
    "VIXCLS": "VIX",
    "DCOILWTICO": "WTI",   # petróleo WTI
    "NFCI": "NFCI"         # Chicago Fed Financial Conditions Index
}


def fred_obs(series_id: str) -> pd.DataFrame:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = dict(
        series_id=series_id,
        api_key=FRED_API_KEY,
        file_type="json",
        observation_start=START_DATE,
        observation_end=END_DATE,
    )
    r = SESSION.get(url, params=params, timeout=60)
    r.raise_for_status()
    obs = r.json().get("observations", [])
    df = pd.DataFrame(obs)[["date", "value"]].rename(columns={"date": "Date", "value": series_id})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    return df


df_fred = None
for sid in FRED_SERIES:
    d = fred_obs(sid)
    df_fred = d if df_fred is None else df_fred.merge(d, on="Date", how="outer")

df_fred = df_fred.sort_values("Date").rename(columns=FRED_SERIES)

if MAKE_DAILY_PANEL:
    df_fred = df_fred.rename(columns={"Date": "Fecha"})
    df_fred = to_daily_ffill(df_fred, "Fecha")
    df_fred = df_fred.rename(columns={"Fecha": "Date"})

df_fred.to_csv(BASE_ORIG / "fred_series.csv", index=False)


# =========================
# 2) Yahoo (sin key) base única + USD/BRL
# =========================
import yfinance as yf

YAHOO = {
    # FX base
    "EURUSD=X": "EURUSD",
    "USDCAD=X": "USDCAD",
    "USDMXN=X": "USDMXN",
    "USDBRL=X": "USDBRL",
    "USDCLP=X": "USDCLP",
    "USDCOP=X": "USDCOP",

    # FX nuevos
    "USDKRW=X": "USDKRW",   # Corea
    "USDINR=X": "USDINR",   # India
    "USDPEN=X": "USDPEN",   # Perú
    "USDPYG=X": "USDPYG",   # Paraguay (a veces Yahoo lo devuelve vacío)
    "NZDUSD=X": "NZDUSD",   # NZD/USD -> para USD/NZD hay que invertir

    # Índices
    "^BVSP": "BOVESPA",
    "^GSPC": "SP500",
    "DX-Y.NYB": "DXY",

    # Commodities
    "ZS=F": "SOY",          # Soja
    "ZR=F": "RICE",         # Arroz (rough rice futures)
    "LE=F": "LIVECATTLE",   # Carne (live cattle)
    "GC=F": "GOLD",         # Oro

    # ETF
    "ILF": "ILF",
}

raw = yf.download(
    list(YAHOO.keys()),
    start=START_DATE,
    end=END_DATE,
    progress=False,
    group_by="ticker",
)

def price_series(t: str) -> pd.Series:
    df_t = raw[t]
    col = "Adj Close" if "Adj Close" in df_t.columns else "Close"
    return df_t[col].rename(t)

prices = pd.concat([price_series(t) for t in YAHOO], axis=1).reset_index()
prices["Date"] = pd.to_datetime(prices["Date"], errors="coerce").dt.normalize()

yahoo_out = pd.DataFrame({"Date": prices["Date"]})

# EURUSD=X es EUR/USD -> queremos USD/EUR
yahoo_out["USD_EUR"] = 1.0 / prices["EURUSD=X"].astype(float)

# FX (USD/moneda local) directos
yahoo_out["USD_CAD"] = prices["USDCAD=X"].astype(float)
yahoo_out["USD_MXN"] = prices["USDMXN=X"].astype(float)
yahoo_out["USD_BRL"] = prices["USDBRL=X"].astype(float)
yahoo_out["USD_CLP"] = prices["USDCLP=X"].astype(float)
yahoo_out["USD_COP"] = prices["USDCOP=X"].astype(float)
yahoo_out["USD_KRW"] = prices["USDKRW=X"].astype(float)
yahoo_out["USD_INR"] = prices["USDINR=X"].astype(float)
yahoo_out["USD_PEN"] = prices["USDPEN=X"].astype(float)
yahoo_out["USD_PYG"] = prices["USDPYG=X"].astype(float)

# Nueva Zelanda: Yahoo suele dar NZD/USD, queremos USD/NZD
yahoo_out["USD_NZD"] = 1.0 / prices["NZDUSD=X"].astype(float)

# Índices
yahoo_out["BOVESPA"] = prices["^BVSP"].astype(float)
yahoo_out["SP500"] = prices["^GSPC"].astype(float)
yahoo_out["DXY_index"] = prices["DX-Y.NYB"].astype(float)

# Commodities
yahoo_out["Soy"] = prices["ZS=F"].astype(float)
yahoo_out["Rice"] = prices["ZR=F"].astype(float)
yahoo_out["LiveCattle"] = prices["LE=F"].astype(float)
yahoo_out["Gold"] = prices["GC=F"].astype(float)

# ETF
yahoo_out["ILF_LatAm"] = prices["ILF"].astype(float)

# (Opcional) chequeo de series que vinieron vacías (útil para USDPYG=X)
# print(yahoo_out.isna().mean().sort_values(ascending=False).head(10))

if MAKE_DAILY_PANEL:
    yahoo_out = yahoo_out.rename(columns={"Date": "Fecha"})
    yahoo_out = to_daily_ffill(yahoo_out, "Fecha")
    yahoo_out = yahoo_out.rename(columns={"Fecha": "Date"})

yahoo_out.to_csv(BASE_ORIG / "yahoo_series.csv", index=False)


# =========================
# 3) PolicyUncertainty (EPU Brasil) (sin key)
# =========================
download_file(
    "https://www.policyuncertainty.com/media/Brazil_Policy_Uncertainty_Data.xlsx",
    BASE_ORIG / "Brazil_Policy_Uncertainty_Data.xlsx"
)


# =========================
# 4) BCU directos (sin key) — con SSL fallback verify=False solo para BCU
# =========================
download_file(
    "https://www.bcu.gub.uy/Estadisticas-e-Indicadores/ComercioExterior_ICB/imp_pais_val.xls",
    BASE_ORIG / "Importaciones_pais_val.xls"
)

download_file(
    "https://www.bcu.gub.uy/Estadisticas-e-Indicadores/ComercioExterior_ICB/exp_ciiu_val.xls",
    BASE_ORIG / "exportaciones_por_producto.xls"
)

download_file(
    "https://www.bcu.gub.uy/Servicios-Financieros-SSF/Series%20IF/tasas.xls",
    BASE_ORIG / "tasas_sistema_bancario.xls"
)

download_file(
    "https://www.bcu.gub.uy/Estadisticas-e-Indicadores/Encuesta-Expectativas-Inflacion/iees05i2.XLS",
    BASE_ORIG / "encuesta_expectativas_inflacion.xls"
)


download_file(
    "https://www.bcu.gub.uy/Estadisticas-e-Indicadores/Balanza%20de%20Pagos/dse_bp_m6_arm_scn.xlsx",
    BASE_ORIG / "balanza_pagos.xlsx"
)

download_file(
    "https://www.bcu.gub.uy/Estadisticas-e-Indicadores/MonedayCredito/Activos-de-Reserva/reservas.xls",
    BASE_ORIG / "Reservas_BCU.xls"
)

download_file(
    "https://www.bcu.gub.uy/Estadisticas-e-Indicadores/ComercioExterior_ICB/exp_pais_val.xls",
    BASE_ORIG / "exp_pais_val.xls"
)

download_file(
    "https://www.bcu.gub.uy/Estadisticas-e-Indicadores/Indice_Cambio_Real/TCRE.xls",
    BASE_ORIG / "TCRE_BCU.xls"
)

# PAM diarios (a veces el link directo funciona así)
download_file(
    "https://www.bcu.gub.uy/Estadisticas-e-Indicadores/MonedayCredito/Principales-Agregados-Monetarios-Diarios/PAM_diarios.xls",
    BASE_ORIG / "PAM_diarios.xls"
)

# Balances monetarios consolidados
download_file(
    "https://www.bcu.gub.uy/Estadisticas-e-Indicadores/Balances%20Consolidados/pmam02d.xls",
    BASE_ORIG / "balances_monetarios_consolidados.xls"
)

download_file(
    "https://bcrdgdcprod.blob.core.windows.net/documents/entorno-internacional/documents/Serie_Historica_Spread_del_EMBI.xlsx",
    BASE_ORIG / "Serie_Historica_Spread_del_EMBI.xlsx"
)
# =========================
# 5) MEF (scraping del xlsx “Descargas”)
# =========================
mef_page = "https://www.gub.uy/ministerio-economia-finanzas/datos-y-estadisticas/estadisticas/informacion-resultados-del-sector-publico"
mef_xlsx = pick_first_xlsx_link(mef_page)
download_file(mef_xlsx, BASE_ORIG / "Resultados_Sector_Publico.xlsx")


# =========================
# 6) ASCOMA (scraping del último “download”)
# =========================
ASCOMA_BASE_1 = "https://ascoma.com.uy/documentos-ascoma/estadisticas-de-ventas-de-autos-y-comerciales-livianos/"
ASCOMA_BASE_CP = "https://ascoma.com.uy/documentos-ascoma/estadisticas-de-ventas-de-autos-y-comerciales-livianos/?cp={cp}"

START_YEAR_AUTODATA = 2015
OUT_DIR = BASE_ORIG  # en tu notebook ya es Path("Bases originales")
RAW_AUTODATA = OUT_DIR / "raw_ascoma_autodata"
OUT_XLSX = OUT_DIR / "venta_autos.xlsx"

RAW_AUTODATA.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTH_MAP = {
    "ENE": 1, "ENERO": 1,
    "FEB": 2, "FEBRERO": 2,
    "MAR": 3, "MARZO": 3,
    "ABR": 4, "ABRIL": 4,
    "MAY": 5, "MAYO": 5,
    "JUN": 6, "JUNIO": 6,
    "JUL": 7, "JULIO": 7,
    "AGO": 8, "AGOSTO": 8,
    "SET": 9, "SEP": 9, "SETIEMBRE": 9, "SEPTIEMBRE": 9,
    "OCT": 10, "OCTUBRE": 10,
    "NOV": 11, "NOVIEMBRE": 11,
    "DIC": 12, "DICIEMBRE": 12,
}

def _to_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()

def _extract_year(s: str) -> int | None:
    m = re.search(r"(20\d{2})", s)
    return int(m.group(1)) if m else None

def _fetch_html(url: str) -> str:
    r = SESSION.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    return r.text

def collect_autodata_download_pages() -> dict[int, str]:
    """Devuelve {año: url_pagina_descarga} para ventas-autodata."""
    found = {}
    for cp in range(1, 30):
        url = ASCOMA_BASE_1 if cp == 1 else ASCOMA_BASE_CP.format(cp=cp)
        html = _fetch_html(url)
        soup = BeautifulSoup(html, "html.parser")

        for a in soup.find_all("a", href=True):
            href = a["href"].strip()
            if "/download/" not in href:
                continue
            if "ventas-autodata" not in href.lower():
                continue
            if "acau" in href.lower():
                continue

            text = a.get_text(" ", strip=True) or ""
            year = _extract_year(text) or _extract_year(href)
            if year is None or year < START_YEAR_AUTODATA:
                continue

            if href.startswith("/"):
                href = "https://ascoma.com.uy" + href

            found[year] = href

        # stop early if we already got a contiguous range down to START_YEAR_AUTODATA
        if found and START_YEAR_AUTODATA in found and len(found) >= (datetime.now().year - START_YEAR_AUTODATA + 1):
            break

    return dict(sorted(found.items()))

def find_direct_file_url(download_page_url: str) -> str:
    """Desde la página /download/... encuentra el link directo a .xls/.xlsx."""
    html = _fetch_html(download_page_url)
    soup = BeautifulSoup(html, "html.parser")

    # primero: links directos a archivos
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if re.search(r"\.(xlsx?|csv)(\?|$)", href, flags=re.I):
            return href if href.startswith("http") else urljoin(download_page_url, href)

    # fallback: algunas veces el archivo está “embebido” como wpdmdl=ID
    m = re.search(r"wpdmdl=(\d+)", html, flags=re.I)
    if m:
        return f"https://ascoma.com.uy/?wpdmdl={m.group(1)}"

    # si no hay nada, devolvemos la misma página (a veces dispara descarga)
    return download_page_url

def download_file(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    with SESSION.get(url, stream=True, timeout=120, headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)

def parse_autodata_excel(file_path: Path, year: int) -> pd.DataFrame:
    """Extrae Fecha (1º de mes) y Total desde la fila 'Total'."""
    ext = file_path.suffix.lower()
    engine = "xlrd" if ext == ".xls" else "openpyxl"

    df = pd.read_excel(file_path, header=None, engine=engine)

    # encontrar fila header con meses
    header_row = None
    for i in range(min(40, len(df))):
        row = [_to_str(v).upper().replace(".", "").strip() for v in df.iloc[i].tolist()]
        hits = sum(1 for v in row if v in MONTH_MAP)
        if hits >= 6 and ("ENE" in row or "ENERO" in row) and ("DIC" in row or "DICIEMBRE" in row):
            header_row = i
            header = row
            break
    if header_row is None:
        raise RuntimeError(f"[{year}] No encontré header de meses en {file_path.name}")

    month_cols, month_nums = [], []
    for j, name in enumerate(header):
        if name in MONTH_MAP:
            month_cols.append(j)
            month_nums.append(MONTH_MAP[name])

    # encontrar fila Total (en col 0)
    total_row = None
    for i, v in enumerate(df.iloc[:, 0].astype(str)):
        if (v or "").strip().lower() == "total":
            total_row = i
            break
    if total_row is None:
        raise RuntimeError(f"[{year}] No encontré fila 'Total' en {file_path.name}")

    totals = pd.to_numeric(df.iloc[total_row, month_cols], errors="coerce")
    out = pd.DataFrame(
    {"Fecha": [datetime(year, m, 1) for m in month_nums], "Total": totals.values}
)

# 1) drop NaNs (meses sin dato en el Excel)
    out = out.dropna(subset=["Total"]).copy()

# 2) no incluir meses futuros (meses que aún no ocurrieron)
    today = datetime.today()
    current_month_start = datetime(today.year, today.month-1, 1)
    out = out[out["Fecha"] <= current_month_start].copy()

    out["Total"] = out["Total"].round(0).astype(int)
    return out

# ---- ejecutar descarga + consolidación ----
print("🔎 Buscando archivos AUTODATA en ASCOMA...")
year_to_download_page = collect_autodata_download_pages()
if not year_to_download_page:
    raise RuntimeError("No encontré links AUTODATA en ASCOMA.")

print(f"✅ Años encontrados: {min(year_to_download_page)}..{max(year_to_download_page)} ({len(year_to_download_page)} archivos)")

parts = []

for year, dl_page in year_to_download_page.items():
    print(f"\nAño {year} -> {dl_page}")
    file_url = find_direct_file_url(dl_page)
    print(f"   Archivo URL: {file_url}")

    # elegir extensión por URL (si no hay, default xls)
    clean = re.sub(r"\?.*$", "", file_url)
    ext = Path(clean).suffix.lower()
    if ext not in [".xls", ".xlsx"]:
        ext = ".xls"

    dest = RAW_AUTODATA / f"autodata_{year}{ext}"
    if not dest.exists():
        download_file(file_url, dest)
        print(f"   Descargado: {dest}")
    else:
        print(f"   Ya existe: {dest}")

    if dest.suffix.lower() in [".xls", ".xlsx"]:
        df_year = parse_autodata_excel(dest, year)
        parts.append(df_year)
        print(f"   Meses extraídos: {len(df_year)}")
    else:
        print(f"   [WARN] No parseo automático para {dest.suffix}")

final_autos = pd.concat(parts, ignore_index=True).sort_values("Fecha").drop_duplicates(["Fecha"], keep="last")
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl", datetime_format="DD/MM/YYYY") as writer:
    final_autos.to_excel(writer, index=False, sheet_name="venta_autos")
    ws = writer.sheets["venta_autos"]

    # Columna A = Fecha (1 = A)
    for cell in ws["A"][1:]:  # saltea header
        cell.number_format = "DD/MM/YYYY"

print("\n✅ Listo:")
print(" - RAW:", RAW_AUTODATA)
print(" - Final:", OUT_XLSX)
print("   Rango:", final_autos["Fecha"].min().date(), "->", final_autos["Fecha"].max().date())
print("   Filas:", len(final_autos))


# =========================
# DONE
# =========================
print("\n✅ Descargas OK\n")
print(" -", BASE_ORIG / "fred_series.csv")
print(" -", BASE_ORIG / "yahoo_series.csv")
print(" -", BASE_ORIG / "Brazil_Policy_Uncertainty_Data.xlsx")
print(" -", BASE_ORIG / "Importaciones_pais_val.xls")
print(" -", BASE_ORIG / "tasas_sistema_bancario.xls")
print(" -", BASE_ORIG / "encuesta_expectativas_inflacion.xls")
print(" -", BASE_ORIG / "Resultados_Sector_Publico_latest.xlsx")
print(" -", BASE_ORIG / "venta_autos.xls")

🔎 Buscando archivos AUTODATA en ASCOMA...
✅ Años encontrados: 2015..2026 (12 archivos)

Año 2015 -> https://ascoma.com.uy/download/ventas-autodata-automoviles-0km-2015/
   Archivo URL: https://ascoma.com.uy/?wpdmdl=369
   Ya existe: Bases originales\raw_ascoma_autodata\autodata_2015.xls
   Meses extraídos: 12

Año 2016 -> https://ascoma.com.uy/download/ventas-autodata-automoviles-0km-2016/
   Archivo URL: https://ascoma.com.uy/?wpdmdl=370
   Ya existe: Bases originales\raw_ascoma_autodata\autodata_2016.xls
   Meses extraídos: 12

Año 2017 -> https://ascoma.com.uy/download/ventas-autodata-automoviles-0km-2017/
   Archivo URL: https://ascoma.com.uy/?wpdmdl=371
   Ya existe: Bases originales\raw_ascoma_autodata\autodata_2017.xls
   Meses extraídos: 12

Año 2018 -> https://ascoma.com.uy/download/ventas-autodata-automoviles-0km-2018/
   Archivo URL: https://ascoma.com.uy/?wpdmdl=372
   Ya existe: Bases originales\raw_ascoma_autodata\autodata_2018.xls
   Meses extraídos: 12

Año 2019 -> http